# ST-Adaptive-Ensemble-RL v5 – refit 3 nhánh trên Train+Val (GPU Kaggle)

Một notebook cho mọi tài khoản; **chỉ đổi biến `ACCOUNT`** ở cell Cấu hình:

| `ACCOUNT` | Dataset | Nhánh nhận kết quả |
|---|---|---|
| `"A"` | GÉANT | `rl_v5_a` |
| `"B"` | SDN + ABILENE | `rl_v5_b` |
| `"SOLO"` | cả 3 dataset trên một tài khoản | `rl_v5_solo` |

`GBDT_ONLY = True`: chỉ refit 2 nhánh GBDT (không cần GPU), ST-WaveFormer giữ dự báo gốc; nhánh kết quả có hậu tố `_gbdt`
(ví dụ `rl_v5_solo_gbdt`), kết quả đánh giá ở `results/refit/gbdt_only/`.

- **Refit:** ST-WaveFormer (E* epoch), XGBoost và LightGBM-Res (số cây = best iteration) huấn luyện lại trên Train+Val;
  scaler, siêu tham số và trọng số stacking giữ nguyên v3. Mã nguồn: nhánh `rl_v5`.
- **GPU:** mỗi run ST-WaveFormer là một tiến trình, 1 run / GPU (run có E* lớn trước). **CPU** song song: XGBoost, LightGBM-Res.
- **Commit ngay:** mỗi (nhánh mô hình, dataset, run) xong là commit + push riêng lên `rl_v5_a` / `rl_v5_b` / `rl_v5_solo`.
- **Đánh giá:** dataset đủ 3 nhánh × 10 run thì chạy `rl_gate/eval_refit.py` -> `results/refit/{ds}_summary.md`.
  Phương án chính chốt trước: `rl_gate_refit_safe`.
- **Kết thúc:** tài khoản xong sau giữ khóa `rl_v5_final_lock`, gộp kết quả vào `rl_v5_results`, tạo `results/refit/summary_all.md`.
- **Bị ngắt / hết 12 giờ:** chạy lại notebook; tự tiếp tục từ nhánh của tài khoản, bỏ qua đơn vị đã có kết quả.

**Trước khi chạy:** Settings, Accelerator: **GPU T4 x2**; Internet: **On**; Add-ons, Secrets: `GITHUB_TOKEN` (quyền ghi repo).
Chạy bằng **Save Version, Save & Run All**.

## 1. Cấu hình

In [ ]:
import os

ACCOUNT = "A"            # "A" = GÉANT, "B" = SDN + ABILENE, "SOLO" = cả 3 dataset: biến duy nhất cần đổi
assert ACCOUNT in ("A", "B", "SOLO"), 'ACCOUNT phải là "A", "B" hoặc "SOLO"'
RUN_IDS = "0-9"          # run r dùng seed 42 + r; đơn vị đã có kết quả sẽ được bỏ qua
GBDT_ONLY = False        # True: chỉ refit XGBoost + LightGBM-Res (CPU, không cần GPU), ST-WaveFormer giữ dự báo gốc

REPO_SLUG   = "Vannt20/network_prediction_v2"
BASE_BRANCH = "rl_v5"    # mã v5 (v3 + rl_gate + refit), không ghi vào nhánh này từ Kaggle
SECRET_NAME = "GITHUB_TOKEN"
GIT_USER_NAME, GIT_USER_MAIL = "Van207", "Nguyenthevan207@gmail.com"
WORKDIR = "/kaggle/working/network_prediction_v2"
BRANCH  = f"rl_v5_{ACCOUNT.lower()}" + ("_gbdt" if GBDT_ONLY else "")

## 2. Môi trường

So phiên bản torch/CUDA với output cùng cell ở tài khoản kia: phải giống nhau.

In [ ]:
import importlib.util, platform, subprocess, sys
missing = [p for p in ["lightgbm", "xgboost"] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import numpy, sklearn, torch, xgboost, lightgbm
print(f"python {platform.python_version()} | torch {torch.__version__} (CUDA {torch.version.cuda}) | numpy {numpy.__version__} | "
      f"sklearn {sklearn.__version__} | xgboost {xgboost.__version__} | lightgbm {lightgbm.__version__} | "
      f"GPU {torch.cuda.device_count()} x {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'} | CPU {os.cpu_count()}")
assert GBDT_ONLY or torch.cuda.device_count() >= 2, "Cần Accelerator = GPU T4 x2"

## 3. GitHub và mã nguồn

Token truyền qua header HTTP ở từng lệnh `git`, không ghi ra đĩa hay output. `logs/` (mô hình gốc, E*, số cây) và
`cache/v3/` (dự báo gốc) nằm sẵn trong repo, không dùng git LFS.

In [ ]:
import base64
from kaggle_secrets import UserSecretsClient

_TOKEN = UserSecretsClient().get_secret(SECRET_NAME)
assert _TOKEN, f"Không đọc được secret {SECRET_NAME}"
_AUTH = base64.b64encode(f"x-access-token:{_TOKEN}".encode()).decode()
_AUTH_ARGS = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {_AUTH}"]


def git(*args, cwd=None, auth=False, check=True, quiet=True):
    """Chỉ in khi lỗi (đã che token)."""
    r = subprocess.run(["git"] + (_AUTH_ARGS if auth else []) + list(args), cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_TOKEN, "***").replace(_AUTH, "***").strip()
    if (not quiet or r.returncode) and out and check:
        print(f"$ git {' '.join(args)}\n{out[-2000:]}")
    if check and r.returncode:
        raise RuntimeError(f"git {args[0]} lỗi (exit {r.returncode})")
    return r


git("config", "--global", "user.name", GIT_USER_NAME)
git("config", "--global", "user.email", GIT_USER_MAIL)
git("config", "--global", "http.postBuffer", "524288000")
REPO_URL = f"https://github.com/{REPO_SLUG}.git"

RESUMED = bool(git("ls-remote", "--heads", REPO_URL, BRANCH, auth=True).stdout.strip())
SRC = BRANCH if RESUMED else BASE_BRANCH
if not os.path.isdir(os.path.join(WORKDIR, ".git")):
    git("clone", "-q", "--depth", "1", "--branch", SRC, REPO_URL, WORKDIR, auth=True)
    git("checkout", "-q", "-B", BRANCH, cwd=WORKDIR)
else:   # chạy lại trong cùng phiên: lấy commit mới nhất của nhánh trên GitHub
    git("checkout", "-q", "-B", BRANCH, cwd=WORKDIR)
    git("pull", "-q", "--ff-only", "origin", SRC, cwd=WORKDIR, auth=True)
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
assert os.path.isdir("cache/v3") and os.path.isdir("logs") and os.path.isfile("training/kaggle_refit.py"), \
    "Thiếu cache/v3, logs/ hoặc mã refit trong bản clone"
print("Tiếp tục từ" if RESUMED else "Bắt đầu từ", SRC, "->", BRANCH, "|", git("log", "--oneline", "-1", cwd=WORKDIR).stdout.strip())

## 4. Huấn luyện và đánh giá

Output chỉ là **một bảng trạng thái**, vẽ lại mỗi 60 giây: việc trên từng GPU (dataset, run, epoch/E*, s/epoch, VRAM),
làn CPU (GBDT), làn đánh giá, số đơn vị đã xong, số commit đã push và 10 sự kiện gần nhất.
Log từng tiến trình: `/kaggle/working/logs_refit/{model}_{ds}_run{r}.log` (ghi đè khi chạy lại).

In [ ]:
from training.kaggle_refit import Refit

job = Refit(ACCOUNT, git, WORKDIR, RUN_IDS, gbdt_only=GBDT_ONLY)
job.train()

## 5. Kết thúc phần của tài khoản này

Kiểm tra đủ kết quả, đánh dấu `status/refit_{account}.json` là `done` và push. Nếu tài khoản kia chưa xong, dừng ở đây.

In [ ]:
job.finish()

## 6. Tổng hợp cuối (chỉ tài khoản xong sau, hoặc SOLO)

Giữ khóa `rl_v5_final_lock`, gộp các nhánh tài khoản vào `rl_v5_results` bằng `git merge --no-ff`, tạo
`results/refit/summary_all.md` và push. Muốn làm lại bước này thì xóa nhánh `rl_v5_final_lock` trên GitHub.

In [ ]:
if job.do_final:
    job.final()
job.summary()

## 7. Trên máy local (sau khi kiểm tra kết quả)

```bash
git fetch origin rl_v5_results
git show origin/rl_v5_results:results/refit/summary_all.md
```

Không có bước tự động merge vào `main`; xóa các nhánh tạm (`rl_v5_a`, `rl_v5_b`, `rl_v5_final_lock`, ...) chỉ khi người dùng quyết định.